In [ ]:
import os
import re
import string
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import nltk

# Download all required NLTK resources (safe to run multiple times)
for resource in ['stopwords', 'wordnet', 'omw-1.4', 'opinion_lexicon']:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Warning: could not download NLTK resource '{resource}': {e}")

from nltk.corpus import stopwords, opinion_lexicon
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Perceptron
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

RANDOM_STATE = 42

print('All libraries loaded successfully.')

In [ ]:
# Load dataset
CSV_PATH = 'Sarcasm_Headlines_Dataset.csv'
df = pd.read_csv(CSV_PATH)

# 1. Column names — confirms no surprises
print(f'\nColumn names  : {list(df.columns)}')

# 2. Shape
print(f'Total rows    : {len(df):,}')
print(f'Total columns : {df.shape[1]}')

# 3. Missing values
print(f'\nMissing values per column:')
print(df.isnull().sum().to_string())

# 4. Class distribution — computed from data
class_counts = df['is_sarcastic'].value_counts().sort_index()
class_pct = df['is_sarcastic'].value_counts(normalize=True).sort_index() * 100

print(f'\nClass distribution:')
print(f'  Not sarcastic (0) : {class_counts[0]:>6,}  ({class_pct[0]:.2f}%)')
print(f'  Sarcastic     (1) : {class_counts[1]:>6,}  ({class_pct[1]:.2f}%)')
print(f'  Total \t: {len(df):>6,}')

In [ ]:
X = df['news_headline'].values
y = df['is_sarcastic'].values

# Step 1: split off 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

# Step 2: split temp equally into validation and test (each 15% of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

# Verify the split sizes and class balance
def split_report(name, y_split):
    n = len(y_split)
    n_sarc = y_split.sum()
    pct = n / len(y) * 100
    print(f'  {name:<12}: {n:>6,} rows ({pct:4.1f}% of total) '
          f'| Sarcastic: {n_sarc:,} ({n_sarc/n*100:.1f}%) '
          f'| Not sarcastic: {n - n_sarc:,} ({(n - n_sarc)/n*100:.1f}%)')

print('SPLIT VERIFICATION')
print('=' * 80)
split_report('Train',      y_train)
split_report('Validation', y_val)
split_report('Test',       y_test)
print('=' * 80)
print(f'Total: {len(y_train)+len(y_val)+len(y_test):,} (should equal {len(y):,})')

In [ ]:
# Initialise NLP tools 
STOP_WORDS = set(stopwords.words('english'))
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_stem(text):
    """Preprocessing with stemming — used by the rule-based method."""
    text   = text.lower()
    text   = re.sub(r'[^a-z\s]', ' ', text)          # remove punctuation/digits
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens                                      # returns a list of tokens

def preprocess_lemma(text):
    """Preprocessing with lemmatisation — used by the ML method (TF-IDF)."""
    text   = text.lower()
    text   = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)                            # returns a clean string


In [ ]:
# Rule Based Lexicon Classifier - Method 1
# Preprocess the data for the rule-based method (stemming)

X_train_stem = [preprocess_stem(t) for t in X_train]
X_val_stem   = [preprocess_stem(t) for t in X_val]
X_test_stem  = [preprocess_stem(t) for t in X_test]
# Load opinion lexicon
POS_WORDS = set(opinion_lexicon.positive())
NEG_WORDS = set(opinion_lexicon.negative())

print(f'Opinion lexicon loaded:')
print(f'  Positive words: {len(POS_WORDS):,}')
print(f'  Negative words: {len(NEG_WORDS):,}')

# Sarcasm indicator words (hand-crafted — rule-based domain knowledge)
# These are hyperbolic, exaggerated or ironic terms typical in sarcastic headlines
SARCASM_INDICATORS = {
    'heroic', 'genius', 'brilliant', 'amazing', 'stunning', 'incredible',
    'unbelievable', 'shocking', 'obviously', 'clearly', 'definitely',
    'totally', 'absolutely', 'literally', 'perfect', 'greatest', 'best',
    'worst', 'terrible', 'awful', 'hilarious', 'weird', 'bizarre',
    'announces', 'reveals', 'confirms', 'admits', 'vows', 'pledges',
    'nation', 'area', 'local', 'man', 'woman', 'report', 'sources',
    'experts', 'scientists', 'study', 'poll', 'survey'
}

print(f'Sarcasm indicator words: {len(SARCASM_INDICATORS)}')

In [ ]:
def compute_rule_features(token_list):
    """
    Compute three rule-based features for a tokenised headline:
      1. Sentiment score (based on positive vs. negative word counts)
      2. Sarcasm indicator count
      3. Positive word count
    """
    if not token_list:
        return 0.0, 0, 0

    pos_count  = sum(1 for t in token_list if t in POS_WORDS)
    neg_count  = sum(1 for t in token_list if t in NEG_WORDS)
    sarc_count = sum(1 for t in token_list if t in SARCASM_INDICATORS)
    total      = len(token_list)

    # Sentiment score formula from Session 5
    sent_score = (pos_count - neg_count) / total
    return sent_score, sarc_count, pos_count

def rule_based_predict(token_lists, sent_threshold, sarc_threshold):
    """
    Label as SARCASTIC (1) if:
      - sentiment score > sent_threshold  (positive language in absurd context)
      OR
      - sarcasm indicator count >= sarc_threshold
    """
    predictions = []
    for tokens in token_lists:
        sent_score, sarc_count, _ = compute_rule_features(tokens)
        pred = 1 if (sent_score > sent_threshold or sarc_count >= sarc_threshold) else 0
        predictions.append(pred)
    return np.array(predictions)


print('Rule-based classifier functions defined.')

In [ ]:
# Threshold tuning on VALIDATION set
# We sweep sentiment and sarcasm indicator thresholds and pick the combination
# with the best F1 score on validation data.

sent_thresholds = np.arange(-0.2, 0.5, 0.05)   # sentiment score thresholds
sarc_thresholds = [1, 2, 3]                      # sarcasm indicator counts

best_f1   = -1
best_sent = None
best_sarc = None
results   = []

for st in sent_thresholds:
    for sc in sarc_thresholds:
        preds = rule_based_predict(X_val_stem, st, sc)
        f1    = f1_score(y_val, preds, zero_division=0)
        acc   = accuracy_score(y_val, preds)
        results.append({'sent_thresh': round(st, 3),
                         'sarc_thresh': sc,
                         'f1': round(f1, 4),
                         'accuracy': round(acc, 4)})
        if f1 > best_f1:
            best_f1   = f1
            best_sent = st
            best_sarc = sc

results_df = pd.DataFrame(results).sort_values('f1', ascending=False)

print('Top 10 threshold combinations (validation set):')
print(results_df.head(10).to_string(index=False))
print(f'\nBest sentiment threshold : {best_sent:.3f}')
print(f'Best sarcasm threshold   : {best_sarc}')
print(f'Best validation F1       : {best_f1:.4f}')

In [ ]:
# Evaluate on TEST set with best thresholds
y_pred_rule = rule_based_predict(X_test_stem, best_sent, best_sarc)

rb_accuracy  = accuracy_score(y_test, y_pred_rule)
rb_precision = precision_score(y_test, y_pred_rule, zero_division=0)
rb_recall    = recall_score(y_test, y_pred_rule, zero_division=0)
rb_f1        = f1_score(y_test, y_pred_rule, zero_division=0)
rb_cm        = confusion_matrix(y_test, y_pred_rule)

print('RULE-BASED CLASSIFIER — TEST SET RESULTS')
print('=' * 45)
print(f'Accuracy  : {rb_accuracy:.4f}  ({rb_accuracy*100:.2f}%)')
print(f'Precision : {rb_precision:.4f}')
print(f'Recall    : {rb_recall:.4f}')
print(f'F1 Score  : {rb_f1:.4f}')
print('\nConfusion Matrix:')
print(rb_cm)
print('\nFull Classification Report:')
print(classification_report(y_test, y_pred_rule,
      target_names=['Not Sarcastic', 'Sarcastic']))

In [ ]:
# TF-IDF Vectorisation - Method 2
# Lemmatise the data for the ML method (TF-IDF)
X_train_lem  = [preprocess_lemma(t) for t in X_train]
X_val_lem    = [preprocess_lemma(t) for t in X_val]
X_test_lem   = [preprocess_lemma(t) for t in X_test]
# Fit on training data ONLY; transform all three splits
tfidf = TfidfVectorizer(
    max_df=0.90,      # ignore terms in >90% of documents (too common)
    min_df=2,         # ignore terms appearing in fewer than 2 documents
    ngram_range=(1, 2),  # unigrams and bigrams
    sublinear_tf=True    # apply log(TF) to dampen very frequent terms
)

X_train_tfidf = tfidf.fit_transform(X_train_lem)
X_val_tfidf   = tfidf.transform(X_val_lem)
X_test_tfidf  = tfidf.transform(X_test_lem)

vocab_size = len(tfidf.vocabulary_)
print(f'TF-IDF matrix shape (train): {X_train_tfidf.shape}')
print(f'TF-IDF matrix shape (val)  : {X_val_tfidf.shape}')
print(f'TF-IDF matrix shape (test) : {X_test_tfidf.shape}')
print(f'Vocabulary size            : {vocab_size:,} terms')

In [ ]:
# Model selection: sweep max_iter on validation set
# Following the Session 8 model-selection pattern:
# build classifiers with different parameters, evaluate on validation, pick best.

iter_values  = [50, 100, 200, 500, 1000, 2000]
val_f1_scores = []
val_acc_scores = []

for n_iter in iter_values:
    clf = Perceptron(max_iter=n_iter, tol=1e-3, random_state=RANDOM_STATE, n_jobs=-1)
    clf.fit(X_train_tfidf, y_train)
    preds   = clf.predict(X_val_tfidf)
    val_f1_scores.append(f1_score(y_val, preds, zero_division=0))
    val_acc_scores.append(accuracy_score(y_val, preds))

best_idx      = int(np.argmax(val_f1_scores))
best_max_iter = iter_values[best_idx]

print('Perceptron model selection (validation F1):')
######
for n, f, a in zip(iter_values, val_f1_scores, val_acc_scores):
    marker = ' ← BEST' if n == best_max_iter else ''
    print(f'  max_iter={n:>5} | val F1={f:.4f} | val Acc={a:.4f}{marker}')
print(f'\nSelected max_iter: {best_max_iter}')

In [ ]:
# Train final Perceptron with best max_iter
best_clf = Perceptron(max_iter=best_max_iter, tol=1e-3,
                      random_state=RANDOM_STATE, n_jobs=-1)
best_clf.fit(X_train_tfidf, y_train)

# Evaluate on TEST set
y_pred_ml = best_clf.predict(X_test_tfidf)

ml_accuracy  = accuracy_score(y_test, y_pred_ml)
ml_precision = precision_score(y_test, y_pred_ml, zero_division=0)
ml_recall    = recall_score(y_test, y_pred_ml, zero_division=0)
ml_f1        = f1_score(y_test, y_pred_ml, zero_division=0)
ml_cm        = confusion_matrix(y_test, y_pred_ml)

print('ML CLASSIFIER (TF-IDF + PERCEPTRON) — TEST SET RESULTS')
print('=' * 55)
print(f'Accuracy  : {ml_accuracy:.4f}  ({ml_accuracy*100:.2f}%)')
print(f'Precision : {ml_precision:.4f}')
print(f'Recall    : {ml_recall:.4f}')
print(f'F1 Score  : {ml_f1:.4f}')
print('\nConfusion Matrix:')
print(ml_cm)
print('\nFull Classification Report:')
print(classification_report(y_test, y_pred_ml,
      target_names=['Not Sarcastic', 'Sarcastic']))

In [ ]:
import random

# Pick a random index from the test set
random_idx      = random.randint(0, len(X_test) - 1)
raw_headline    = X_test[random_idx]          # original unprocessed headline
true_label      = y_test[random_idx]          # integer ground truth (0 or 1)
true_name       = 'Sarcastic' if true_label == 1 else 'Not Sarcastic'

# Rule-Based Prediction
# Preprocess with lemmatisation (token list) for lexicon lookup

lem_tokens      = preprocess_lemma(raw_headline)

# compute_rule_features() already defined in Section 5 — reuse directly
sent_score, sarc_count, pos_count = compute_rule_features(lem_tokens)

# Classify using the best thresholds found on the validation set
rb_pred_label   = 1 if (sent_score > best_sent or sarc_count >= best_sarc) else 0
rb_pred_name    = 'Sarcastic' if rb_pred_label == 1 else 'Not Sarcastic'
rb_correct      = (rb_pred_label == true_label)
rb_result       = "✓ CORRECT" if rb_correct else "✗ WRONG"

# ── Step 3: ML Prediction (TF-IDF + Perceptron) ──────────────────────────────
# Preprocess with lemmatisation (string) for TF-IDF — same pipeline as training
lem_string      = preprocess_lemma(raw_headline)

# Transform using the fitted TF-IDF vectoriser — never refit on new data
tfidf_vector    = tfidf.transform([lem_string])

# Predict using the best Perceptron found during model selection
ml_pred_label   = best_clf.predict(tfidf_vector)[0]
ml_pred_name    = 'Sarcastic' if ml_pred_label == 1 else 'Not Sarcastic'
ml_correct      = (ml_pred_label == true_label)
ml_result       = "✓ CORRECT" if ml_correct else "✗ WRONG"

# ── Step 4: Print a clear comparison ─────────────────────────────────────────
print('=' * 65)
print(f'Test headline index : {random_idx}')
print(f'\nHeadline  : "{raw_headline}"')
print(f'True class: {true_name}')
print()

print('── Rule-Based (Lexicon) ──────────────────────────────────────')
print(f'  Preprocessed tokens : {lem_tokens}')
print(f'  Sentiment score     : {sent_score:.4f}  '
      f'(threshold = {best_sent:.3f})')
print(f'  Indicator count     : {sarc_count}  '
      f'(threshold = {best_sarc})')
print(f'  Prediction          : {rb_pred_name}')
print(f'  Result              : {rb_result}')
print()

print('── ML (TF-IDF + Perceptron) ──────────────────────────────────')
print(f'  Preprocessed string : "{lem_string}"')

# Show the top 5 TF-IDF terms with non-zero weight for this headline
feature_names   = tfidf.get_feature_names_out()
tfidf_scores    = tfidf_vector.toarray()[0]
nonzero_idx     = tfidf_scores.nonzero()[0]
top_terms       = sorted(nonzero_idx,
                         key=lambda i: tfidf_scores[i],
                         reverse=True)[:5]
top_terms_str   = ', '.join(
    f'"{feature_names[i]}" ({tfidf_scores[i]:.3f})'
    for i in top_terms
)
print(f'  Top TF-IDF terms    : {top_terms_str}')
print(f'  Prediction          : {ml_pred_name}')
print(f'  Result              : {ml_result}')
print()

print('── Summary ───────────────────────────────────────────────────')
print(f'  True class          : {true_name}')
print(f'  Rule-Based pred     : {rb_pred_name}  {rb_result}')
print(f'  ML pred             : {ml_pred_name}  {ml_result}')
print('=' * 65)